# Lab04.2 — Logitech C270 USB Webcam

**Objective:** capture and display real camera images in Jupyter on Zybo Legacy / PYNQ 3.0.1.

Connect the Logitech C270. The tested device is `/dev/video0`. Run the setup cell once, then execute the acquisition cells as often as necessary.

## 1. Camera setup

The helper opens the device through V4L2, requests 640 × 480, discards 30 warm-up frames and releases the camera with `finally`. No new OpenCV installation is needed.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image

CAMERA_DEVICE = "/dev/video0"
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
WARMUP_FRAMES = 30

@contextmanager
def camera_session():
    # Open the Logitech C270 through the Linux V4L2 interface.
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open camera: " + CAMERA_DEVICE)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError("Camera warm-up frame failed.")
            time.sleep(0.05)
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def capture_frame():
    with camera_session() as cap:
        ok, frame = cap.read()
        if not ok or frame is None:
            raise RuntimeError("No camera frame captured.")
    return frame

def show_bgr(img):
    ok, encoded = cv2.imencode(".jpg", img)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    display(Image(data=encoded.tobytes()))

## 2. Capture and display one frame

Place a colored marker in view. Run this cell again to capture a new frame.

In [ ]:
frame = capture_frame()
print("OpenCV version:", cv2.__version__)
print("Captured frame (H, W, C):", frame.shape)
show_bgr(frame)

## 3. Short sequential capture (optional)

Acquire five frames, each separated by approximately 0.2 seconds. This is not a live video stream.

In [ ]:
frames = []
with camera_session() as cap:
    for i in range(5):
        ok, frame = cap.read()
        if not ok or frame is None:
            raise RuntimeError("Frame {} failed.".format(i + 1))
        frames.append(frame.copy())
        time.sleep(0.2)
print("Captured frames:", len(frames))
show_bgr(frames[-1])

## Student check

1. Record the actual image resolution and OpenCV version.
2. Run the single-frame capture twice to confirm that the camera is reopened correctly.
3. Explain the purpose of warm-up and `cap.release()`.

If the camera cannot open, stop other notebooks/kernels that may still hold `/dev/video0`. Use `!fuser -v /dev/video0` only for troubleshooting.